# Día 2b — Cargar datos externos (CSV real + SQLite)
### Preparación entrevista técnica Ebury (Treasury/ALM)

Hasta ahora has practicado con DataFrames construidos a mano — limpios por diseño. Esto es distinto a propósito: vas a cargar un CSV real con los problemas típicos de datos exportados de sistemas reales (formatos de fecha mezclados, mayúsculas inconsistentes, nulos, una fila duplicada), y una base de datos SQLite para practicar SQL de verdad.

**Importante**: antes de mirar la estructura de los datos aquí abajo, imagina que no sabes nada de ellos — así es como te va a llegar el lunes. El primer paso siempre es explorar, nunca asumir.

## Parte 0 — Construir un DataFrame a partir de una lista

No descartes que, en vez de un archivo, simplemente te den (o pegues tú) una lista de Python — de diccionarios, de tuplas, o de listas — y tengas que convertirla en DataFrame antes de trabajar con ella. Practica las tres formas más comunes.

In [112]:
import pandas as pd
import numpy as np
from fontTools.varLib.mutator import curr


### 1. Lista de diccionarios (la más habitual, y la más segura)
Cada diccionario es una fila; las claves se convierten automáticamente en columnas. Ventaja: no dependes del orden, y es tolerante a que falte alguna clave en algún registro.

In [113]:
lista_dicts = [
    {"divisa": "USD", "importe": 120000, "contraparte": "Acme Corp"},
    {"divisa": "EUR", "importe": -45000, "contraparte": "Beta PYME"},
    {"divisa": "GBP", "importe": 30000, "contraparte": "Gamma Ltd"},
]

df_desde_dicts = pd.DataFrame(lista_dicts)
print(df_desde_dicts)


  divisa  importe contraparte
0    USD   120000   Acme Corp
1    EUR   -45000   Beta PYME
2    GBP    30000   Gamma Ltd


In [114]:
# Si a algún diccionario le falta una clave, pandas rellena con NaN automáticamente — no rompe nada
lista_dicts_incompleta = [
    {"divisa": "USD", "importe": 120000},
    {"divisa": "EUR", "importe": -45000, "contraparte": "Beta PYME"},
]
print(pd.DataFrame(lista_dicts_incompleta))


  divisa  importe contraparte
0    USD   120000         NaN
1    EUR   -45000   Beta PYME


### 2. Lista de tuplas (o de listas) — tienes que dar los nombres de columna tú mismo
Aquí pandas no sabe cómo se llama cada "campo", así que hay que indicarlo con el parámetro `columns`. Es fácil que se te olvide bajo presión — practícalo para que salga automático.

In [115]:
lista_tuplas = [
    ("USD", 120000, "Acme Corp"),
    ("EUR", -45000, "Beta PYME"),
    ("GBP", 30000, "Gamma Ltd"),
]

df_desde_tuplas = pd.DataFrame(lista_tuplas, columns=["divisa", "importe", "contraparte"])
print(df_desde_tuplas)


  divisa  importe contraparte
0    USD   120000   Acme Corp
1    EUR   -45000   Beta PYME
2    GBP    30000   Gamma Ltd


### 3. Lista de listas, con los nombres de columna en una lista aparte
Muy típico si te copian/pegan datos "en bruto" — por ejemplo, la salida de una consulta a una API o de otro sistema que te dan como filas + cabecera por separado.

In [116]:
filas = [
    ["USD", 120000, "Acme Corp"],
    ["EUR", -45000, "Beta PYME"],
    ["GBP", 30000, "Gamma Ltd"],
]
columnas = ["divisa", "importe", "contraparte"]

df_desde_listas = pd.DataFrame(filas, columns=columnas)
print(df_desde_listas)


  divisa  importe contraparte
0    USD   120000   Acme Corp
1    EUR   -45000   Beta PYME
2    GBP    30000   Gamma Ltd


### Ejercicio 0.1
Te dan estos datos "en bruto", tal como te los podrían pasar en la entrevista (una lista de diccionarios con algunas claves inconsistentes). Conviértelos en DataFrame y comprueba qué columnas y cuántos nulos genera pandas automáticamente.

In [117]:
datos_brutos = [
    {"ccy": "USD", "amt": 100000, "cpty": "Acme Corp", "maturity": "2026-09-10"},
    {"ccy": "EUR", "amt": -50000, "cpty": "Beta PYME", "maturity": "2026-10-01"},
    {"ccy": "GBP", "amt": 30000, "maturity": "2026-09-20"},   # sin "cpty"
    {"ccy": "USD", "amt": 75000, "cpty": "Acme Corp"},         # sin "maturity"
]

# TODO: convierte esto en DataFrame y comprueba nulos
df_bruto =pd.DataFrame(datos_brutos)
null_data=df_bruto["ccy"][df_bruto.isna().any(axis=1)]
print(null_data)
print(f"There are: ", len(null_data), "raws with not a number. Those columns are: ", null_data)

print(df_bruto)



2    GBP
3    USD
Name: ccy, dtype: str
There are:  2 raws with not a number. Those columns are:  2    GBP
3    USD
Name: ccy, dtype: str
   ccy     amt       cpty    maturity
0  USD  100000  Acme Corp  2026-09-10
1  EUR  -50000  Beta PYME  2026-10-01
2  GBP   30000        NaN  2026-09-20
3  USD   75000  Acme Corp         NaN


**Solución 0.1**

In [118]:
df_bruto = pd.DataFrame(datos_brutos)
print(df_bruto)
print()
print(df_bruto.isna().sum())


   ccy     amt       cpty    maturity
0  USD  100000  Acme Corp  2026-09-10
1  EUR  -50000  Beta PYME  2026-10-01
2  GBP   30000        NaN  2026-09-20
3  USD   75000  Acme Corp         NaN

ccy         0
amt         0
cpty        1
maturity    1
dtype: int64


> Fíjate que las claves de los diccionarios en este ejercicio (`ccy`, `amt`, `cpty`) no coinciden con el nombre "bonito" que quizás esperarías (`divisa`, `importe`, `contraparte`). Esto pasa constantemente con datos reales — el primer paso después de convertir a DataFrame suele ser renombrar columnas con `.rename(columns={...})`, que ya practicaste el Día 1.

---
## Parte 1 — Cargar y explorar un CSV desconocido

Ahora sí, pasamos a la otra fuente de datos posible: un archivo. Asegúrate de que `transacciones_tesoreria.csv` está en la misma carpeta que este notebook (o ajusta la ruta).

In [173]:
import pandas as pd
import numpy as np

df = pd.read_csv("transacciones_tesoreria.csv")
df.head()


,TransID,Currency,Amount,CounterpartyID,MaturityDate,Notes
0,1001,JPY,36965.83,1,09-02-2026,NaN
1,1002,usd,-181419.83,1,2026-11-29,NaN
2,1003,GBP,43017.94,1,06/12/2026,NaN
3,1004,usd,-131790.35,1,09-23-2026,NaN
4,1005,usd,-173979.36,4,11-16-2026,NaN


### Ejercicio 1.1 — Diagnóstico inicial
Antes de tocar nada, responde estas preguntas con código (esto es lo primero que deberías hacer siempre con un dataset nuevo el lunes):
1. ¿Cuántas filas y columnas tiene?
2. ¿Qué tipo de dato tiene cada columna? ¿Alguna te parece "incorrecta" (por ejemplo, una fecha guardada como texto)?
3. ¿Cuántos valores nulos hay por columna?
4. ¿Hay filas duplicadas?

In [171]:
# TODO: resuelve las 4 preguntas con código

filas=df.shape[0]
columnas=df.shape[1]
print(f"filas is:",filas)
print(f"Columnas is: ",columnas)
print(df.dtypes)
null_values_column=df.isna().sum()
print(f"null_values")
duplicate_rows=df.duplicated()
duplicate_rows=duplicate_rows[duplicate_rows==True]
print("--------- Duplicated rows-------")
print(duplicate_rows)

# Q1: raws and columns
print(f"Q1: shape is: ", np.shape(df))
rawsdf=np.shape(df)[0]
columnsdf=np.shape(df)[1]

# Q2
print(df.head())
print(df.dtypes)
print(df.isna().head())

print(f"Q2: The currencies with NaN are: ",pd.unique(df["Currency"][df.isna().any(axis=1)]))
print(f"Q2: It seems wrong the type of the dates (as it is str) and also Notes as it is always NaN instead of float64")

# Q3
print(f"Q3: Null values by column: ",df.isna().sum()) # Valores nulos por columna

# Q4
duplicateRaws=[]
for i in range(len(df)-1):
    if (df.iloc[i,:]==df.iloc[i+1,:]).all():
        duplicateRaws.append(i+1)
print(f"Q4: These are the duplicates: ",duplicateRaws)

duplicateRaws=df.duplicated()
print(f"Q4: These are the duplicates: ",duplicateRaws.sum()," and the raw is: ", duplicateRaws[duplicateRaws==True])

filas is: 41
Columnas is:  6
TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes             float64
dtype: object
null_values
--------- Duplicated rows-------
40    True
dtype: bool
Q1: shape is:  (41, 6)
   TransID Currency     Amount  CounterpartyID MaturityDate  Notes
0     1001      JPY   36965.83               1   09-02-2026    NaN
1     1002      usd -181419.83               1   2026-11-29    NaN
2     1003      GBP   43017.94               1   06/12/2026    NaN
3     1004      usd -131790.35               1   09-23-2026    NaN
4     1005      usd -173979.36               4   11-16-2026    NaN
TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes             float64
dtype: object
   TransID  Currency  Amount  CounterpartyID  MaturityDate  Notes
0    False     False   False           False         False   True
1    F

**Solución 1.1**

In [121]:
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
print("Nulos por columna:")
print(df.isna().sum())
print()
print("Filas duplicadas:", df.duplicated().sum())


Shape: (41, 6)

TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes             float64
dtype: object

Nulos por columna:
TransID            0
Currency           1
Amount             1
CounterpartyID     0
MaturityDate       0
Notes             41
dtype: int64

Filas duplicadas: 1


Deberías haber visto: `MaturityDate` es de tipo `object` (texto), no fecha — hay que convertirla. Hay nulos en `Amount`, `Currency` y toda la columna `Notes`. Y hay al menos una fila duplicada.

### Ejercicio 1.2 — Limpieza
1. Elimina las filas duplicadas.
2. Elimina la columna `Notes` (está completamente vacía, no aporta nada).
3. Normaliza `Currency` a mayúsculas (fíjate que hay "usd" y "USD" mezclados — para pandas son valores distintos aunque para nosotros signifiquen lo mismo).
4. Convierte `MaturityDate` a fecha real. Pista: mira los valores únicos de la columna primero — vas a encontrar más de un formato mezclado, así que un único `pd.to_datetime()` simple probablemente falle o dé resultados incorrectos en parte de las filas.

In [172]:
# TODO: los 4 pasos de limpieza
dfcopy=df.copy()
dfcopy=dfcopy.drop_duplicates()
dfcopy=dfcopy.drop(columns="Notes")
dfcopy["Currency"]=dfcopy["Currency"].str.upper()
dfcopy["MaturityDate"]=pd.to_datetime(dfcopy["MaturityDate"],format="mixed")

df=pd.DataFrame(pd.read_csv("transacciones_tesoreria.csv"))

# Q1:
df_limpio=df.copy()
duplicateRaws=df.duplicated()
df_limpio=df_limpio[~duplicateRaws]
print(df_limpio)

# Q2:
df_limpio=df_limpio.drop(columns="Notes")
print(df_limpio.head())

# Q3:
unique_corr=df_limpio["Currency"].unique()
print(unique_corr)
df_limpio["Currency"] = df_limpio["Currency"].str.upper()

# Q4:
df_clean_dates=[pd.to_datetime(df_limpio["MaturityDate"][i],format="mixed", dayfirst=True) for i in range(len(df_limpio))]
df_limpio["MaturityDate"] = pd.to_datetime(df_limpio["MaturityDate"], format="mixed", dayfirst=True)


print(df_limpio.dtypes)
print(df_limpio.isna().sum())
df_limpio.head()


    TransID Currency     Amount  CounterpartyID MaturityDate  Notes
0      1001      JPY   36965.83               1   09-02-2026    NaN
1      1002      usd -181419.83               1   2026-11-29    NaN
2      1003      GBP   43017.94               1   06/12/2026    NaN
3      1004      usd -131790.35               1   09-23-2026    NaN
4      1005      usd -173979.36               4   11-16-2026    NaN
5      1006      EUR        NaN               3   2026-10-06    NaN
6      1007      GBP  186252.81               3   09-11-2026    NaN
7      1008      GBP  123358.94               1   12/01/2027    NaN
8      1009      GBP  -78154.49               3   29/12/2026    NaN
9      1010      usd -160931.15               3   31/12/2026    NaN
10     1011      JPY   73693.21               1   25/11/2026    NaN
11     1012      GBP  -23939.00               3   18/02/2027    NaN
12     1013      NaN -151184.71               5   2026-09-20    NaN
13     1014      usd   -1929.24               2 

,TransID,Currency,Amount,CounterpartyID,MaturityDate
0,1001,JPY,36965.83,1,2026-02-09
1,1002,USD,-181419.83,1,2026-11-29
2,1003,GBP,43017.94,1,2026-12-06
3,1004,USD,-131790.35,1,2026-09-23
4,1005,USD,-173979.36,4,2026-11-16


**Solución 1.2**

In [123]:
df_limpio = df.copy()

# 1. Delete duplicados
df_limpio = df_limpio.drop_duplicates()

# 2. Elimitate column
df_limpio = df_limpio.drop(columns=["Notes"])

# 3. Normalizar texto: convert to capital letter
df_limpio["Currency"] = df_limpio["Currency"].str.upper()

# 4. Fechas con formato mixto — pandas moderno permite format="mixed" para esto exactamente
df_limpio["MaturityDate"] = pd.to_datetime(df_limpio["MaturityDate"], format="mixed", dayfirst=False, errors="coerce")

print(df_limpio.dtypes)
print(df_limpio.isna().sum())
df_limpio.head()


TransID                    int64
Currency                     str
Amount                   float64
CounterpartyID             int64
MaturityDate      datetime64[us]
dtype: object
TransID           0
Currency          1
Amount            1
CounterpartyID    0
MaturityDate      0
dtype: int64


,TransID,Currency,Amount,CounterpartyID,MaturityDate
0,1001,JPY,36965.83,1,2026-09-02
1,1002,USD,-181419.83,1,2026-11-29
2,1003,GBP,43017.94,1,2026-06-12
3,1004,USD,-131790.35,1,2026-09-23
4,1005,USD,-173979.36,4,2026-11-16


> **Nota importante**: `errors="coerce"` convierte a `NaT` (fecha nula) cualquier valor que no pueda interpretarse, en vez de romper todo el proceso con un error. Es una decisión consciente: prefieres perder unas pocas filas con fecha rara, identificables después, a que se caiga todo el script. Si te preguntan por esto en la entrevista, es una buena respuesta: mencionar que lo hiciste a propósito y que revisarías cuántas filas quedaron con `NaT` antes de continuar.

In [124]:
# Cuántas fechas no se pudieron interpretar (si las hay)
print("Fechas no interpretables:", df_limpio["MaturityDate"].isna().sum())


Fechas no interpretables: 0


### Ejercicio 1.3
Con los datos ya limpios, calcula el importe total por divisa (ignora las filas con `Amount` nulo).

In [176]:
# TODO
total_curr_sum=df.dropna(subset="Amount").groupby(["Currency"])["Amount"].sum().reset_index()
print("total_curr_sum")
print(total_curr_sum)







print(df_limpio.head())

# Without groupby:
final_results=[] # Initialize the list
for i in df_limpio["Currency"].unique():
    subset=df_limpio[df_limpio["Currency"]==i]
    sumtot=np.sum(df_limpio["Amount"][df_limpio["Currency"]==i])
    final_results.append({
        "Currency": i,
        "Total Sum": sumtot
    })
final_results_df=pd.DataFrame(final_results).sort_values("Total Sum",ascending=False)
final_results_df=final_results_df.dropna()
print(final_results_df)
df_eur=df_limpio[(df_limpio["Currency"]=="EUR") & (df_limpio["Amount"]>0)]
print(df_eur.head())

# Groupby
solution1=df_limpio.dropna(subset=["Amount"]).groupby("Currency")["Amount"].sum().reset_index()
solution2=df_limpio.dropna(subset=["Amount"]).groupby("Currency")["Amount"].agg(["sum","mean","count"]).reset_index()

print("Gap")
print(solution1)
print(solution2)


total_curr_sum
  Currency     Amount
0      EUR -221374.90
1      GBP  -35901.31
2      JPY  350667.26
3      USD -121911.53
4      eur  130890.61
5      usd -605225.23
   TransID Currency     Amount  CounterpartyID MaturityDate
0     1001      JPY   36965.83               1   2026-02-09
1     1002      USD -181419.83               1   2026-11-29
2     1003      GBP   43017.94               1   2026-12-06
3     1004      USD -131790.35               1   2026-09-23
4     1005      USD -173979.36               4   2026-11-16
  Currency  Total Sum
0      JPY  350667.26
2      GBP  -35901.31
3      EUR  -90484.29
1      USD -595346.41
    TransID Currency     Amount  CounterpartyID MaturityDate
17     1018      EUR   65008.91               1   2026-09-28
23     1024      EUR  110053.13               4   2027-03-08
24     1025      EUR  175799.58               3   2027-12-01
34     1035      EUR  131495.00               1   2026-08-11
Gap
  Currency     Amount
0      EUR  -90484.29
1      G

**Solución 1.3**

In [126]:
resumen = df_limpio.dropna(subset=["Amount"]).groupby("Currency")["Amount"].sum().reset_index()
print(resumen)


  Currency     Amount
0      EUR  -90484.29
1      GBP  -35901.31
2      JPY  350667.26
3      USD -595346.41


---
## Parte 2 — Cargar y consultar una base de datos SQLite

Ahora practica con `tesoreria.db`, un archivo de base de datos real (no un CSV) con dos tablas: `transactions` y `counterparties`. Esto simula que te dan acceso a un archivo de base de datos, en vez de un CSV suelto.

In [127]:
import sqlite3

conn = sqlite3.connect("tesoreria.db")

# Primero, averigua qué tablas tiene la base de datos — no lo des por hecho
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tablas)

# Get shape of the tables
for table_name in tablas["name"]:
    table_sql=pd.read_sql(f"SELECT * FROM {table_name};", conn)
    print(f"Table: ",{table_name}, "shape:" ,{table_sql.shape})
    print(table_sql.dtypes)


             name
0    transactions
1  counterparties
Table:  {'transactions'} shape: {(41, 6)}
TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes              object
dtype: object
Table:  {'counterparties'} shape: {(5, 3)}
CounterpartyID    int64
Name                str
ClientType          str
dtype: object


In [128]:
# Explora cada tabla con un SELECT * LIMIT antes de trabajar con ella a fondo
print(pd.read_sql("SELECT * FROM transactions LIMIT 5;", conn))

# Get the full table in DataFrame format
dfsql=pd.read_sql("SELECT * FROM transactions;",conn)
print(dfsql.dtypes)
print(dfsql.shape)


   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
TransID             int64
Currency              str
Amount            float64
CounterpartyID      int64
MaturityDate          str
Notes              object
dtype: object
(41, 6)


In [129]:
print(pd.read_sql("SELECT * FROM counterparties;", conn))
print(pd.read_sql("SELECT * from transactions LIMIT 5;",conn))


   CounterpartyID         Name ClientType
0               1    Acme Corp  Corporate
1               2    Beta PYME       PYME
2               3    Gamma Ltd       PYME
3               4     Delta SA  Corporate
4               5  Epsilon Inc       PYME
   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None


### Ejercicio 2.1
Escribe una consulta SQL (usando `pd.read_sql`) que devuelva el importe total (`Amount`) por divisa (`Currency`), ordenado de mayor a menor importe absoluto.

In [178]:
# TODO: escribe la query SQL, y luego descomenta las dos líneas siguientes para ejecutarla
query="""
SELECT Currency, SUM(Amount) as sumtotal
FROM transactions
WHERE Amount IS NOT NULL
GROUP BY Currency
ORDER BY ABS(sumtotal) DESC
LIMIT 10
"""
print(pd.read_sql(query,conn))



  Currency   sumtotal
0      usd -605225.23
1      JPY  350667.26
2      EUR -221374.90
3      NaN -151184.71
4      eur  130890.61
5      USD -121911.53
6      GBP  -35901.31
  Currency  sum_amount
0      usd  -605225.23
1      JPY   350667.26
2      EUR  -221374.90
3      NaN  -151184.71
4      eur   130890.61
5      USD  -121911.53
6      GBP   -35901.31


**Solución 2.1**

In [131]:
query ="""
SELECT Currency, SUM(Amount) as total_amount
FROM transactions
WHERE Amount IS NOT NULL
GROUP BY Currency
ORDER BY ABS(SUM(Amount)) DESC
"""

resultado = pd.read_sql(query, conn)
print(resultado)


  Currency  total_amount
0      usd    -605225.23
1      JPY     350667.26
2      EUR    -221374.90
3      NaN    -151184.71
4      eur     130890.61
5      USD    -121911.53
6      GBP     -35901.31


### Ejercicio 2.2 — JOIN en SQL
Escribe una consulta que una `transactions` con `counterparties`, y devuelva el importe total por `ClientType` (Corporate vs PYME).

In [180]:
# TODO
query="""
SELECT SUM(t.Amount) as sumtotal, c.ClientType
FROM transactions t
LEFT JOIN counterparties c ON c.CounterpartyID=t.CounterpartyID
WHERE t.Amount IS NOT NULL
GROUP BY c.ClientType
ORDER BY sumtotal DESC
LIMIT 10
"""
print(pd.read_sql(query,conn))


    sumtotal ClientType
0 -150448.90  Corporate
1 -503590.91       PYME
  ClientType  suma_total
0  Corporate  -150448.90
1       PYME  -503590.91
    sumtotal ClientType
0 -150448.90  Corporate
1 -503590.91       PYME
  ClientType  suma_total
0  Corporate  -150448.90
1       PYME  -503590.91


**Solución 2.2**

In [133]:
query_join = """
SELECT c.ClientType, SUM(t.Amount) as total_amount
FROM transactions t
LEFT JOIN counterparties c ON t.CounterpartyID = c.CounterpartyID
WHERE t.Amount IS NOT NULL
GROUP BY c.ClientType
"""
resultado_join = pd.read_sql(query_join, conn)
print(resultado_join)

conn.close()  # buena práctica: cerrar la conexión cuando termines


  ClientType  total_amount
0  Corporate    -150448.90
1       PYME    -503590.91


### Ejercicio 2.3 — Lo mismo, pero con DuckDB directamente sobre el CSV
Hasta ahora has lanzado SQL contra un archivo SQLite (`tesoreria.db`). **DuckDB** es otro motor SQL, pero con una diferencia clave: puede ejecutar consultas **directamente sobre un DataFrame de pandas que ya tienes en memoria**, sin necesidad de una base de datos ni una conexión — solo `import duckdb` y `duckdb.sql("SELECT ... FROM nombre_del_dataframe")`, usando el nombre de la variable Python como si fuera una tabla.

Repite el Ejercicio 2.1 (importe total por divisa) pero usando DuckDB sobre `df_limpio` en vez de SQLite.

In [185]:
import duckdb

# TODO
query="""
SELECT Currency, SUM(Amount) as total_amount
FROM df_limpio
WHERE Amount IS NOT NULL
GROUP BY Currency
ORDER BY SUM(Amount) DESC
"""
resultado_duckdb = duckdb.sql(query).df()
print(resultado_duckdb)


  Currency    tot_sum
0      JPY  350667.26
1      GBP  -35901.31
2      EUR  -90484.29
3      NaN -151184.71
4      USD -595346.41
  Currency  total_amount
0      JPY     350667.26
1      GBP     -35901.31
2      EUR     -90484.29
3      NaN    -151184.71
4      USD    -595346.41


**Solución 2.3**

In [135]:
resultado_duckdb = duckdb.sql("""
    SELECT Currency, SUM(Amount) as total_amount
    FROM df_limpio
    WHERE Amount IS NOT NULL
    GROUP BY Currency
    ORDER BY ABS(SUM(Amount)) DESC
""").df()
print(resultado_duckdb)


  Currency  total_amount
0      USD    -595346.41
1      JPY     350667.26
2      NaN    -151184.71
3      EUR     -90484.29
4      GBP     -35901.31


In [136]:
conn = sqlite3.connect("tesoreria.db")

### Ejercicio 2.4 — `WHERE` + `ORDER BY` + `LIMIT`
Encuentra las **5 transacciones de mayor importe** (en valor absoluto) que vencen **después del 1 de octubre de 2026**. Devuelve `TransID`, `Currency`, `Amount`, `MaturityDate`.

In [189]:
# TODO
query="""
SELECT TransID, Currency, Amount, MaturityDate
FROM transactions
WHERE MaturityDate>"2026-10-01"
ORDER BY ABS(Amount) DESC
LIMIT 5
"""
print(pd.read_sql(query,conn))


   TransID Currency     Amount         MaturityDate
0     1023      JPY  187833.85  2026-11-15 00:00:00
1     1015      EUR -186244.59  2026-10-20 00:00:00
2     1031      GBP -181909.08  2026-12-31 00:00:00
3     1002      usd -181419.83  2026-11-29 00:00:00
4     1025      eur  175799.58  2027-01-12 00:00:00
   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
2.4. RESULT:
   TransID Currency     Amount         MaturityDate
0     1023      JPY  187833.85  2026-11-15 00:00:00
1     1015      EUR -186244.59  2026-10-20 00:00:00
2     1031      GBP -181909.08  2026-12-31 00:00:00
3     1002      usd -18

**Solución 2.4**

In [141]:
query_top5 = """
SELECT TransID, Currency, Amount, MaturityDate
FROM transactions
WHERE MaturityDate > '2026-10-01'
ORDER BY ABS(Amount) DESC
LIMIT 5
"""
resultado_top5 = pd.read_sql(query_top5, conn)
print(resultado_top5)

   TransID Currency     Amount         MaturityDate
0     1023      JPY  187833.85  2026-11-15 00:00:00
1     1015      EUR -186244.59  2026-10-20 00:00:00
2     1031      GBP -181909.08  2026-12-31 00:00:00
3     1002      usd -181419.83  2026-11-29 00:00:00
4     1025      eur  175799.58  2027-01-12 00:00:00


### Ejercicio 2.5 — `GROUP BY` + `HAVING` (con `JOIN`)
Cuenta cuántas transacciones tiene cada contraparte (usa `JOIN` con `counterparties` para mostrar el nombre, no solo el ID), y quédate **solo con las que tienen más de 3 transacciones**.

Pista: `HAVING` es como `WHERE`, pero se aplica *después* de agrupar — no puedes usar `WHERE COUNT(*) > 3` porque `WHERE` se evalúa antes de que exista el conteo.

In [192]:
# TODO

print(pd.read_sql("SELECT * FROM transactions LIMIT 5;",conn))
print(pd.read_sql("SELECT * FROM counterparties LIMIT 5;",conn))

query25="""
SELECT c.ClientType,c.Name, COUNT(t.CounterpartyID) as num_transactions
FROM transactions t
LEFT JOIN counterparties c ON t.CounterpartyID=c.CounterpartyID
GROUP BY c.Name
HAVING COUNT(t.CounterpartyID)>3
ORDER BY num_transactions DESC
"""
solution25=pd.read_sql(query25,conn)
print("Solution 2.5. is:")
print(solution25)

   num_transactions         Name
0                13    Acme Corp
1                11    Gamma Ltd
2                 6    Beta PYME
3                 5  Epsilon Inc
4                 5     Delta SA
   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
   CounterpartyID         Name ClientType
0               1    Acme Corp  Corporate
1               2    Beta PYME       PYME
2               3    Gamma Ltd       PYME
3               4     Delta SA  Corporate
4               5  Epsilon Inc       PYME
Solution 2.5. is:
  ClientType         Name  num_transactions
0  Corporate    Acme Corp                13


**Solución 2.5**

In [104]:
query_count = """
SELECT c.Name, COUNT(*) as num_transacciones
FROM transactions t
JOIN counterparties c ON t.CounterpartyID = c.CounterpartyID
GROUP BY c.Name
HAVING COUNT(*) > 3
ORDER BY num_transacciones DESC
"""
resultado_count = pd.read_sql(query_count, conn)
print(resultado_count)

conn.close()  # buena práctica: cerrar la conexión cuando termines

          Name  num_transacciones
0    Acme Corp                 13
1    Gamma Ltd                 12
2    Beta PYME                  6
3  Epsilon Inc                  5
4     Delta SA                  5


---
## Parte 3 — Combinar tablas en pandas: `merge()`

Ya uniste `transactions` con `counterparties` en SQL (Ejercicios 2.2 y 2.5) con `JOIN`. Ahora practica el equivalente en pandas: `.merge()`. La lógica es la misma — necesitas una columna en común entre las dos tablas (`CounterpartyID`) — solo cambia la sintaxis.

```python
df.merge(otra_tabla, on="columna_en_comun", how="left")
```
`how` funciona igual que en SQL: `"left"` = `LEFT JOIN` (te quedas con todas las filas de `df`, aunque no encuentren pareja), `"inner"` = `JOIN` normal (solo filas que casan en ambas tablas), `"right"` y `"outer"` también existen, igual que en SQL.

Como cerraste `conn` al final del Ejercicio 2.5, hace falta reabrirla para cargar las tablas como DataFrames.

In [145]:
conn = sqlite3.connect("tesoreria.db")

df_transactions = pd.read_sql("SELECT * FROM transactions", conn)
counterparties = pd.read_sql("SELECT * FROM counterparties", conn)

print(df_transactions.head())
print(counterparties.head())

   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
   CounterpartyID         Name ClientType
0               1    Acme Corp  Corporate
1               2    Beta PYME       PYME
2               3    Gamma Ltd       PYME
3               4     Delta SA  Corporate
4               5  Epsilon Inc       PYME


### Ejercicio 3.1 — el equivalente en pandas del Ejercicio 2.2
Reproduce el resultado del Ejercicio 2.2, pero sin SQL: usa `.merge()` para unir `df_transactions` con `counterparties` por `CounterpartyID`, y calcula el importe total (`Amount`) por `ClientType` (ignora las filas con `Amount` nulo).

In [194]:
# TODO: tu turno -- resuélvelo sin mirar la solución
result31=df_transactions.dropna(subset=["Amount"]).merge(counterparties, on="CounterpartyID", how="left").groupby(["ClientType"])["Amount"].agg(["sum","mean"]).reset_index()
print(result31)


  ClientType        sum          mean
0  Corporate -150448.90  -8358.272222
1       PYME -503590.91 -22890.495909
  ClientType     Amount
0  Corporate -150448.90
1       PYME -503590.91


**Solución 3.1**

In [153]:
merged = df_transactions.merge(counterparties, on="CounterpartyID", how="left")

resultado31 = merged.dropna(subset=["Amount"]).groupby("ClientType")["Amount"].sum().reset_index()
print(resultado31)

  ClientType     Amount
0  Corporate -150448.90
1       PYME -503590.91


### Ejercicio 3.2 — el equivalente en pandas del Ejercicio 2.5 (la trampa del `HAVING`)
Reproduce el resultado del Ejercicio 2.5: cuenta cuántas transacciones tiene cada contraparte (usa `.merge()` para mostrar el `Name`, no solo el `CounterpartyID`), y quédate solo con las que tienen más de 3 transacciones.

Pista importante: en pandas **no existe un `HAVING`** — ni falta que hace. Como `.groupby()` ya te devuelve un DataFrame/Series normal, simplemente lo filtras después con una condición booleana, igual que harías con cualquier otro DataFrame. Es la misma idea que `HAVING` en SQL (filtrar *después* de agregar), pero sin necesitar una palabra clave especial — en pandas es el mismo `[...]` que ya usas siempre.

In [199]:
# TODO: tu turno -- resuélvelo sin mirar la solución
conn=sqlite3.connect("tesoreria.db")
print(pd.read_sql("SELECT * from transactions;",conn).head())
print(pd.read_sql("SELECT * from counterparties;",conn).head())

result32=df_transactions.dropna(subset=["Amount"]).merge(counterparties, on="CounterpartyID", how="left").groupby("Name")["TransID"].count().reset_index(name="num_transactions").sort_values(by=["num_transactions"], ascending= False)

print(result32)


   TransID Currency     Amount  CounterpartyID         MaturityDate Notes
0     1001      JPY   36965.83               1  2026-09-02 00:00:00  None
1     1002      usd -181419.83               1  2026-11-29 00:00:00  None
2     1003      GBP   43017.94               1  2026-06-12 00:00:00  None
3     1004      usd -131790.35               1  2026-09-23 00:00:00  None
4     1005      usd -173979.36               4  2026-11-16 00:00:00  None
   CounterpartyID         Name ClientType
0               1    Acme Corp  Corporate
1               2    Beta PYME       PYME
2               3    Gamma Ltd       PYME
3               4     Delta SA  Corporate
4               5  Epsilon Inc       PYME
          Name  num_transactions
0    Acme Corp                13
4    Gamma Ltd                11
1    Beta PYME                 6
2     Delta SA                 5
3  Epsilon Inc                 5
          Name  num_transactions
0    Acme Corp                13
1    Beta PYME                 6
2     D

**Solución 3.2**

In [157]:
conteo = merged.groupby("Name")["TransID"].count().reset_index(name="num_transacciones")

# El "HAVING" es simplemente esto: filtrar el resultado ya agregado, como cualquier DataFrame
resultado32 = conteo[conteo["num_transacciones"] > 3].sort_values("num_transacciones", ascending=False)
print(resultado32)

conn.close()  # buena práctica: cerrar la conexión cuando termines

          Name  num_transacciones
0    Acme Corp                 13
4    Gamma Ltd                 12
1    Beta PYME                  6
2     Delta SA                  5
3  Epsilon Inc                  5


---
## Resumen del Día 2b
- Cargar un CSV real y **desconfiar por defecto** de su limpieza: revisar shape, dtypes, nulos y duplicados antes de hacer nada más.
- Manejar formatos de fecha mezclados con `format="mixed"` y `errors="coerce"`, y verificar cuántas filas quedaron sin poder convertirse.
- Conectarte a un archivo SQLite con `sqlite3.connect()`, listar sus tablas antes de asumir su estructura, y usar `pd.read_sql()` para traer resultados directamente como DataFrame.
- Escribir SQL con `GROUP BY`, `JOIN`, y `ORDER BY` — la misma lógica que ya practicaste en pandas, ahora en el otro lenguaje.
- Lo mismo con DuckDB directamente sobre un DataFrame, sin necesidad de una base de datos separada.
- `.merge()` en pandas como equivalente directo de `JOIN` en SQL — y por qué no existe un `HAVING` en pandas: simplemente filtras el resultado ya agregado con `[...]`, como cualquier otro DataFrame.

Con esto ya has practicado los dos escenarios de origen de datos más probables para el lunes: **archivo suelto (CSV/Excel)** y **base de datos (SQLite/SQL)**. Cuando lo tengas trabajado, seguimos con el Día 3.